# 民声智理 · Google Colab GPU 算力节点部署套件

本 Notebook 用于在 Google Colab（推荐 **L4 24GB** / **A100 40GB** / **T4 16GB** GPU）或任意 Linux GPU 环境上一键拉起：

1. **System-2 认知大模型服务**（端口 `8132`，100% OpenAI 兼容 `/v1/chat/completions` 与 `/v1/models`）
2. **BAAI/bge-m3 向量嵌入服务**（端口 `8133`，100% OpenAI 兼容 `/v1/embeddings` 与 `/v1/models`，1024 维）
3. **公网/内网穿透隧道**（支持 `cloudflared` 零配置 HTTPS 临时隧道或 `tailscale` 加密内网）
4. **强制 Bearer Token 鉴权**（未手动指定时运行时自动生成高强度随机密钥）

> 请先在 Colab 菜单栏确认：**代码执行程序 (Runtime) → 更改运行时类型 (Change runtime type) → GPU**。

## Step 1：检查 GPU 型号与可用显存 (`nvidia-smi`)

脚本将根据检测到的显存自动匹配最优默认模型：
- **A100 80GB+**：`Qwen/Qwen2.5-32B-Instruct`
- **A100 40GB**：`Qwen/Qwen2.5-14B-Instruct`
- **L4 24GB**：`Qwen/Qwen2.5-14B-Instruct-AWQ`
- **T4 16GB**：`Qwen/Qwen2.5-7B-Instruct-AWQ`

In [ ]:
!nvidia-smi

import shutil, subprocess
if shutil.which("nvidia-smi"):
    info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip()
    print("\n检测到 GPU:", info)
else:
    print("\n⚠️ 未检测到 NVIDIA GPU，请在 Colab 切换至 GPU 运行时。")

## Step 2：安装依赖与准备 `serve_colab.py`

- 选择 LLM 后端依赖：`vllm`（推荐，高吞吐）或 `llama-cpp-python`（适合 GGUF 权重）
- 安装 Embedding 服务依赖：`sentence-transformers`、`fastapi`、`uvicorn`、`huggingface_hub`
- 按需安装隧道工具：`cloudflared`（默认，无需注册即可生成 HTTPS 临时地址）或 `tailscale`
- 若当前目录尚未放置 `serve_colab.py`，可在下方提示时直接上传仓库中的 `scripts/colab/serve_colab.py`。

In [ ]:
#@title 安装推理后端、Embedding 服务与隧道组件
INSTALL_BACKEND = "vllm" #@param ["vllm", "llama-cpp"]
INSTALL_TUNNEL = "cloudflared" #@param ["cloudflared", "tailscale", "none"]

import os, pathlib, shutil, subprocess, sys

base_pkgs = ["sentence-transformers", "fastapi", "uvicorn", "huggingface_hub", "pydantic"]
if INSTALL_BACKEND == "vllm":
    base_pkgs.append("vllm")
else:
    base_pkgs.append("llama-cpp-python")

print("正在安装 Python 依赖:", ", ".join(base_pkgs))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *base_pkgs])

if INSTALL_TUNNEL == "cloudflared" and not shutil.which("cloudflared"):
    print("正在安装 cloudflared...")
    subprocess.check_call([
        "bash", "-lc",
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb && dpkg -i /tmp/cloudflared.deb && rm -f /tmp/cloudflared.deb"
    ])
elif INSTALL_TUNNEL == "tailscale" and not shutil.which("tailscale"):
    print("正在安装 tailscale...")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://tailscale.com/install.sh | sh"])

# 检查 serve_colab.py 是否已存在于工作区
candidates = [
    pathlib.Path("serve_colab.py"),
    pathlib.Path("scripts/colab/serve_colab.py"),
    pathlib.Path("/content/serve_colab.py"),
]
found_script = next((p for p in candidates if p.is_file()), None)
if found_script:
    print(f"✅ 已定位启动脚本: {found_script.resolve()}")
else:
    print("未检测到 serve_colab.py，若在 Google Colab 中请通过文件上传窗口上传 scripts/colab/serve_colab.py：")
    try:
        from google.colab import files
        uploaded = files.upload()
        if "serve_colab.py" in uploaded:
            print("✅ 已上传 serve_colab.py")
    except ImportError:
        print("当前非 Colab 环境，请确保 scripts/colab/serve_colab.py 位于工作目录中。")

## Step 3：配置参数（Token、模型选择、隧道模式）

- **`API_KEY`**：留空则自动通过 `secrets.token_urlsafe(24)` 生成高强度随机 Bearer Token（切勿在公开 Notebook 中硬编码真实密钥）。
- **`LLM_MODEL`**：留空则根据 Step 1 检测到的 GPU 显存自动推荐对应量化的 Qwen2.5 模型。
- **`TUNNEL_MODE`**：Colab 公网访问默认选 `cloudflared`；若已登录 Tailscale 内网组网可选 `tailscale`。

In [ ]:
#@title 配置 Colab 算力节点参数
LLM_BACKEND = "vllm" #@param ["vllm", "llama-cpp"]
LLM_MODEL = "" #@param {type:"string"}
EMBED_MODEL = "BAAI/bge-m3" #@param {type:"string"}
TUNNEL_MODE = "cloudflared" #@param ["cloudflared", "tailscale", "none"]
API_KEY = "" #@param {type:"string"}
LLM_PORT = 8132 #@param {type:"integer"}
EMBED_PORT = 8133 #@param {type:"integer"}
EMBED_DEVICE = "auto" #@param ["auto", "cuda", "cpu"]

import os, secrets

resolved_key = API_KEY.strip() or os.environ.get("COLAB_API_KEY", "").strip() or secrets.token_urlsafe(24)
os.environ["COLAB_API_KEY"] = resolved_key

masked = f"****{resolved_key[-4:]}" if len(resolved_key) >= 4 else "****"
print(f"✅ 参数已就绪:")
print(f"  • LLM_BACKEND  : {LLM_BACKEND}")
print(f"  • LLM_MODEL    : {LLM_MODEL or '(按 GPU 显存自动推荐)'}")
print(f"  • EMBED_MODEL  : {EMBED_MODEL} (device={EMBED_DEVICE})")
print(f"  • TUNNEL_MODE  : {TUNNEL_MODE}")
print(f"  • PORTS        : LLM={LLM_PORT}, EMBED={EMBED_PORT}")
print(f"  • BEARER_TOKEN : 已设置 ({masked}，启动后将在下方打印完整接入配置)")

## Step 4：拉起 `serve_colab.py` 并获取系统接入配置

执行本单元后将依次完成：
1. 在 `8133` 端口拉起 `BAAI/bge-m3` OpenAI 兼容向量服务；
2. 在 `8132` 端口拉起 System-2 LLM 服务（首次下载权重约需 1~3 分钟）；
3. 打开隧道并输出可直接复制粘贴到 **「民声智理后台 → 算力集群管理」** 的 `SYSTEM_TWO` Endpoint URL、`EMBEDDING` Endpoint URL、Bearer Token 与模型名称。

> 保持本单元处于运行状态即可持续为「民声智理」提供远程 GPU 算力。

In [ ]:
#@title 启动双引擎算力节点并打印接入面板配置
import os, pathlib, signal, subprocess, sys

candidates = [
    pathlib.Path("serve_colab.py"),
    pathlib.Path("scripts/colab/serve_colab.py"),
    pathlib.Path("/content/serve_colab.py"),
]
script_path = next((p for p in candidates if p.is_file()), None)
if not script_path:
    raise FileNotFoundError("未找到 serve_colab.py，请先完成 Step 2 上传或克隆仓库。")

if "COLAB_API_KEY" not in os.environ:
    raise RuntimeError("请先执行 Step 3 生成或配置 COLAB_API_KEY。")

cmd = [
    sys.executable,
    str(script_path),
    "--llm-backend", LLM_BACKEND,
    "--embed-model", EMBED_MODEL,
    "--llm-port", str(LLM_PORT),
    "--embed-port", str(EMBED_PORT),
    "--tunnel", TUNNEL_MODE,
    "--embed-device", EMBED_DEVICE,
]
if LLM_MODEL.strip():
    cmd.extend(["--llm-model", LLM_MODEL.strip()])

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    print("\n收到中断信号，正在通知 serve_colab.py 清理子进程...")
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=10)

## Step 5：用完即释与安全注意事项

1. **用完即释**：研判演示或批量任务结束后，点击 Step 4 左侧停止按钮中断服务，随后在 Colab 菜单选择 **「代码执行程序 → 断开连接并删除运行时」**，避免持续消耗计算单元（CU）。
2. **后台清理**：回到「民声智理后台 → 多站点与算力管理」面板，停用或删除已关闭的临时 Colab 节点，防止系统持续探活。
3. **密钥安全**：不要保存包含 Step 4 输出（含 Bearer Token 与临时隧道 URL）的 `.ipynb` 副本到公开代码库。